<a href="https://colab.research.google.com/github/Ni777-ctr/eletro-gestor.viabilidade/blob/main/logica_python_eletro_gestor_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

:

In [ ]:
# ============================================================

# ELETROGESTOR - MOTOR CENTRAL DE GESTÃO

# Backend completo em arquivo único

#

# Recursos:

# - FastAPI

# - SQLite

# - Obras / WL

# - Viabilidade

# - Pré-APR com validade automática

# - Materiais

# - Retirado / Instalado / Devolvido

# - Reconciliação automática

# - Equipes

# - Veículos

# - Programação inteligente

# - Inventário

# - Faturamento

# - Dashboard

# - Importação Excel

# - Consolidação de materiais

# - Integração preparada para SIGEO Helper

#

# INSTALAÇÃO:

# pip install fastapi uvicorn sqlalchemy pandas openpyxl python-multipart

#

# EXECUTAR:

# uvicorn main:app --reload

#

# ACESSAR:

# http://127.0.0.1:8000/docs

# ============================================================

import os
import shutil
import datetime

from typing import Optional, List

import pandas as pd

from fastapi import (
FastAPI,
HTTPException,
Depends,
UploadFile,
File
)

from fastapi.middleware.cors import CORSMiddleware

from pydantic import BaseModel

from sqlalchemy import (
create_engine,
Column,
Integer,
String,
Float,
Boolean,
Date,
DateTime,
Text,
ForeignKey
)

from sqlalchemy.orm import (
declarative_base,
sessionmaker,
Session,
relationship
)

# ============================================================

# CONFIGURAÇÕES

# ============================================================

NOME_SISTEMA = "ELETROGESTOR"

VERSAO = "1.0.0"

DATABASE_URL = "sqlite:///./eletrogestor.db"

PASTA_UPLOADS = "uploads"

PASTA_SIGEO = "sigeo_import"

VALIDADE_PRE_APR = 45

ALERTA_PRE_APR = 5

os.makedirs(PASTA_UPLOADS, exist_ok=True)

os.makedirs(PASTA_SIGEO, exist_ok=True)

# ============================================================

# BANCO DE DADOS

# ============================================================

engine = create_engine(
DATABASE_URL,
connect_args={
"check_same_thread": False
}
)

SessionLocal = sessionmaker(
autocommit=False,
autoflush=False,
bind=engine
)

Base = declarative_base()

def get_db():

```
db = SessionLocal()

try:

    yield db

finally:

    db.close()
```

# ============================================================

# MODELOS DO BANCO

# ============================================================

# ============================================================

# OBRAS

# ============================================================

class Obra(Base):

```
__tablename__ = "obras"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

wl = Column(
    String,
    unique=True,
    index=True,
    nullable=False
)

projeto = Column(
    String,
    nullable=False
)

circuito = Column(
    String,
    nullable=True
)

endereco = Column(
    String,
    nullable=True
)

cidade = Column(
    String,
    nullable=True
)

tipo_intervencao = Column(
    String,
    nullable=True
)

status = Column(
    String,
    default="NOVA"
)

equipe = Column(
    String,
    nullable=True
)

veiculo = Column(
    String,
    nullable=True
)

data_programada = Column(
    Date,
    nullable=True
)

percentual_execucao = Column(
    Float,
    default=0
)

criado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow
)

atualizado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow,
    onupdate=datetime.datetime.utcnow
)

viabilidade = relationship(
    "Viabilidade",
    back_populates="obra",
    uselist=False,
    cascade="all, delete-orphan"
)

pre_aprs = relationship(
    "PreAPR",
    back_populates="obra",
    cascade="all, delete-orphan"
)

materiais = relationship(
    "MaterialObra",
    back_populates="obra",
    cascade="all, delete-orphan"
)

inventarios = relationship(
    "Inventario",
    back_populates="obra",
    cascade="all, delete-orphan"
)
```

# ============================================================

# VIABILIDADE

# ============================================================

class Viabilidade(Base):

```
__tablename__ = "viabilidades"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

obra_id = Column(
    Integer,
    ForeignKey("obras.id"),
    unique=True,
    nullable=False
)

responsavel = Column(
    String,
    nullable=False
)

data_inspecao = Column(
    Date,
    nullable=False
)

status = Column(
    String,
    default="PENDENTE"
)

vistoria_manobra_realizada = Column(
    Boolean,
    default=False
)

data_vistoria_manobra = Column(
    Date,
    nullable=True
)

observacoes_manobra = Column(
    Text,
    nullable=True
)

observacoes_gerais = Column(
    Text,
    nullable=True
)

criado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow
)

atualizado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow,
    onupdate=datetime.datetime.utcnow
)

obra = relationship(
    "Obra",
    back_populates="viabilidade"
)
```

# ============================================================

# PRE APR

# ============================================================

class PreAPR(Base):

```
__tablename__ = "pre_apr"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

obra_id = Column(
    Integer,
    ForeignKey("obras.id"),
    nullable=False
)

responsavel = Column(
    String,
    nullable=False
)

data_elaboracao = Column(
    Date,
    nullable=False
)

validade_dias = Column(
    Integer,
    default=VALIDADE_PRE_APR
)

status = Column(
    String,
    default="VALIDA"
)

observacoes = Column(
    Text,
    nullable=True
)

criado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow
)

obra = relationship(
    "Obra",
    back_populates="pre_aprs"
)
```

# ============================================================

# MATERIAIS

# ============================================================

class MaterialObra(Base):

```
__tablename__ = "materiais_obra"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

obra_id = Column(
    Integer,
    ForeignKey("obras.id"),
    nullable=False
)

codigo = Column(
    String,
    index=True,
    nullable=False
)

descricao = Column(
    String,
    nullable=False
)

aplicacao = Column(
    String,
    nullable=True
)

unidade = Column(
    String,
    default="UN"
)

quantidade_prevista = Column(
    Float,
    default=0
)

quantidade_retirada = Column(
    Float,
    default=0
)

quantidade_instalada = Column(
    Float,
    default=0
)

quantidade_devolvida = Column(
    Float,
    default=0
)

criado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow
)

obra = relationship(
    "Obra",
    back_populates="materiais"
)
```

# ============================================================

# EQUIPES

# ============================================================

class Equipe(Base):

```
__tablename__ = "equipes"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

nome = Column(
    String,
    nullable=False
)

codigo = Column(
    String,
    unique=True,
    nullable=False
)

disponivel = Column(
    Boolean,
    default=True
)

certificacao_valida = Column(
    Boolean,
    default=True
)

especialidade = Column(
    String,
    nullable=True
)

cidade_base = Column(
    String,
    nullable=True
)
```

# ============================================================

# VEICULOS

# ============================================================

class Veiculo(Base):

```
__tablename__ = "veiculos"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

placa = Column(
    String,
    unique=True,
    nullable=False
)

tipo = Column(
    String,
    nullable=True
)

disponivel = Column(
    Boolean,
    default=True
)
```

# ============================================================

# INVENTARIO

# ============================================================

class Inventario(Base):

```
__tablename__ = "inventarios"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

obra_id = Column(
    Integer,
    ForeignKey("obras.id"),
    nullable=False
)

foto_antes = Column(
    String,
    nullable=True
)

foto_depois = Column(
    String,
    nullable=True
)

responsavel = Column(
    String,
    nullable=True
)

status = Column(
    String,
    default="PENDENTE"
)

observacoes = Column(
    Text,
    nullable=True
)

criado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow
)

obra = relationship(
    "Obra",
    back_populates="inventarios"
)
```

# ============================================================

# FATURAMENTO

# ============================================================

class Faturamento(Base):

```
__tablename__ = "faturamentos"

id = Column(
    Integer,
    primary_key=True,
    index=True
)

obra_id = Column(
    Integer,
    ForeignKey("obras.id"),
    nullable=False
)

valor_previsto = Column(
    Float,
    default=0
)

valor_executado = Column(
    Float,
    default=0
)

valor_faturado = Column(
    Float,
    default=0
)

status = Column(
    String,
    default="PENDENTE"
)

criado_em = Column(
    DateTime,
    default=datetime.datetime.utcnow
)
```

# ============================================================

# SCHEMAS

# ============================================================

class ObraCreate(BaseModel):

```
wl: str

projeto: str

circuito: Optional[str] = None

endereco: Optional[str] = None

cidade: Optional[str] = None

tipo_intervencao: Optional[str] = None
```

class ViabilidadeCreate(BaseModel):

```
responsavel: str

data_inspecao: datetime.date

status: str = "CONCLUIDA"

vistoria_manobra_realizada: bool = False

data_vistoria_manobra: Optional[datetime.date] = None

observacoes_manobra: Optional[str] = None

observacoes_gerais: Optional[str] = None
```

class PreAPRCreate(BaseModel):

```
responsavel: str

data_elaboracao: datetime.date

validade_dias: int = 45

observacoes: Optional[str] = None
```

class MaterialCreate(BaseModel):

```
codigo: str

descricao: str

aplicacao: Optional[str] = None

unidade: str = "UN"

quantidade_prevista: float = 0

quantidade_retirada: float = 0

quantidade_instalada: float = 0

quantidade_devolvida: float = 0
```

class MovimentoMaterial(BaseModel):

```
quantidade: float
```

class EquipeCreate(BaseModel):

```
nome: str

codigo: str

disponivel: bool = True

certificacao_valida: bool = True

especialidade: Optional[str] = None

cidade_base: Optional[str] = None
```

class VeiculoCreate(BaseModel):

```
placa: str

tipo: Optional[str] = None

disponivel: bool = True
```

class ProgramacaoCreate(BaseModel):

```
equipe_id: int

veiculo_id: int

data_programada: datetime.date
```

class InventarioCreate(BaseModel):

```
responsavel: Optional[str] = None

foto_antes: Optional[str] = None

foto_depois: Optional[str] = None

observacoes: Optional[str] = None
```

class FaturamentoCreate(BaseModel):

```
valor_previsto: float = 0

valor_executado: float = 0

valor_faturado: float = 0

status: str = "PENDENTE"
```

# ============================================================

# MOTOR PRE APR

# ============================================================

def verificar_pre_apr(data_elaboracao, validade_dias=45):

```
hoje = datetime.date.today()

dias_decorridos = (
    hoje - data_elaboracao
).days

dias_restantes = (
    validade_dias - dias_decorridos
)

if dias_restantes < 0:

    status = "VENCIDA"

elif dias_restantes <= ALERTA_PRE_APR:

    status = "PROXIMA_DO_VENCIMENTO"

else:

    status = "VALIDA"

return {

    "data_elaboracao": data_elaboracao,

    "validade_dias": validade_dias,

    "dias_decorridos": dias_decorridos,

    "dias_restantes": dias_restantes,

    "status": status
}
```

# ============================================================

# MOTOR MATERIAIS

# ============================================================

def consolidar_materiais(lista_materiais):

```
materiais = {}

for item in lista_materiais:

    codigo = str(item.codigo).strip()

    if codigo not in materiais:

        materiais[codigo] = {

            "codigo": codigo,

            "descricao": item.descricao,

            "aplicacao": item.aplicacao,

            "unidade": item.unidade,

            "previsto": 0,

            "retirado": 0,

            "instalado": 0,

            "devolvido": 0
        }

    materiais[codigo]["previsto"] += (
        item.quantidade_prevista or 0
    )

    materiais[codigo]["retirado"] += (
        item.quantidade_retirada or 0
    )

    materiais[codigo]["instalado"] += (
        item.quantidade_instalada or 0
    )

    materiais[codigo]["devolvido"] += (
        item.quantidade_devolvida or 0
    )

resultado = []

for codigo, material in materiais.items():

    saldo = (
        material["retirado"]
        - material["instalado"]
        - material["devolvido"]
    )

    diferenca_previsto = (
        material["previsto"]
        - material["instalado"]
    )

    if saldo == 0:

        status = "CORRETO"

    elif saldo > 0:

        status = "MATERIAL_PENDENTE"

    else:

        status = "DIVERGENCIA"

    material["saldo"] = saldo

    material["diferenca_previsto"] = (
        diferenca_previsto
    )

    material["status"] = status

    resultado.append(material)

return resultado
```

# ============================================================

# RECONCILIAR MATERIAL

# ============================================================

def reconciliar_material(material):

```
retirado = (
    material.quantidade_retirada or 0
)

instalado = (
    material.quantidade_instalada or 0
)

devolvido = (
    material.quantidade_devolvida or 0
)

saldo = (
    retirado
    - instalado
    - devolvido
)

if saldo == 0:

    status = "CORRETO"

elif saldo > 0:

    status = "MATERIAL_PENDENTE"

else:

    status = "DIVERGENCIA"

return {

    "codigo": material.codigo,

    "descricao": material.descricao,

    "previsto": (
        material.quantidade_prevista
    ),

    "retirado": retirado,

    "instalado": instalado,

    "devolvido": devolvido,

    "saldo": saldo,

    "status": status
}
```

# ============================================================

# MOTOR DE EQUIPES

# ============================================================

def calcular_score_equipe(

```
equipe,

cidade_obra=None,

especialidade_necessaria=None
```

):

```
score = 0

if equipe.disponivel:

    score += 30

else:

    return 0

if equipe.certificacao_valida:

    score += 30

else:

    return 0

if (
    especialidade_necessaria
    and equipe.especialidade
):

    if (
        especialidade_necessaria.lower()
        in equipe.especialidade.lower()
    ):

        score += 25

else:

    score += 10

if cidade_obra and equipe.cidade_base:

    if (
        cidade_obra.lower()
        == equipe.cidade_base.lower()
    ):

        score += 15

return score
```

# ============================================================

# MOTOR DE PROGRAMAÇÃO

# ============================================================

def recomendar_equipe(

```
obra,

equipes
```

):

```
recomendacoes = []

for equipe in equipes:

    score = calcular_score_equipe(

        equipe=equipe,

        cidade_obra=obra.cidade,

        especialidade_necessaria=(
            obra.tipo_intervencao
        )
    )

    if score > 0:

        recomendacoes.append({

            "equipe_id": equipe.id,

            "codigo": equipe.codigo,

            "nome": equipe.nome,

            "score": score
        })

recomendacoes.sort(

    key=lambda x: x["score"],

    reverse=True
)

return recomendacoes
```

# ============================================================

# MOTOR INVENTÁRIO

# ============================================================

def verificar_inventario(inventarios):

```
if not inventarios:

    return {

        "status": "PENDENTE",

        "motivo": (
            "Inventário não encontrado"
        )
    }

inventario = inventarios[-1]

if not inventario.foto_antes:

    return {

        "status": "PENDENTE",

        "motivo": (
            "Foto antes não cadastrada"
        )
    }

if not inventario.foto_depois:

    return {

        "status": "PENDENTE",

        "motivo": (
            "Foto depois não cadastrada"
        )
    }

return {

    "status": "CONCLUIDO",

    "motivo": "Inventário completo"
}
```

# ============================================================

# MOTOR CENTRAL DA OBRA

# ============================================================

def calcular_status_obra(

```
obra,

db
```

):

```
pendencias = []


# --------------------------------------------------------
# VIABILIDADE
# --------------------------------------------------------

if not obra.viabilidade:

    pendencias.append(
        "Viabilidade não cadastrada"
    )

else:

    if (
        obra.viabilidade.status
        != "CONCLUIDA"
    ):

        pendencias.append(
            "Viabilidade não concluída"
        )


# --------------------------------------------------------
# PRE APR
# --------------------------------------------------------

pre_apr_valida = False

if obra.pre_aprs:

    ultima_pre_apr = obra.pre_aprs[-1]

    resultado_pre_apr = verificar_pre_apr(

        ultima_pre_apr.data_elaboracao,

        ultima_pre_apr.validade_dias
    )

    ultima_pre_apr.status = (
        resultado_pre_apr["status"]
    )

    if (
        resultado_pre_apr["status"]
        == "VALIDA"
    ):

        pre_apr_valida = True

    else:

        pendencias.append(

            f"Pré-APR {resultado_pre_apr['status']}"
        )

else:

    pendencias.append(
        "Pré-APR não cadastrada"
    )


# --------------------------------------------------------
# EQUIPE
# --------------------------------------------------------

if not obra.equipe:

    pendencias.append(
        "Equipe não definida"
    )


# --------------------------------------------------------
# VEICULO
# --------------------------------------------------------

if not obra.veiculo:

    pendencias.append(
        "Veículo não definido"
    )


# --------------------------------------------------------
# MATERIAIS
# --------------------------------------------------------

if not obra.materiais:

    pendencias.append(
        "Materiais não cadastrados"
    )

else:

    materiais_pendentes = []

    for material in obra.materiais:

        saldo = (

            material.quantidade_retirada

            - material.quantidade_instalada

            - material.quantidade_devolvida
        )

        if saldo < 0:

            materiais_pendentes.append(

                material.codigo
            )

    if materiais_pendentes:

        pendencias.append(

            "Divergência de materiais"
        )


# --------------------------------------------------------
# EXECUÇÃO
# --------------------------------------------------------

if obra.percentual_execucao <= 0:

    status_execucao = "NAO_INICIADA"

elif obra.percentual_execucao < 100:

    status_execucao = "EM_EXECUCAO"

else:

    status_execucao = "CONCLUIDA"


# --------------------------------------------------------
# INVENTARIO
# --------------------------------------------------------

resultado_inventario = (

    verificar_inventario(

        obra.inventarios
    )
)


# --------------------------------------------------------
# DEFINIR STATUS FINAL
# --------------------------------------------------------

if obra.percentual_execucao >= 100:

    if (

        resultado_inventario["status"]

        == "CONCLUIDO"
    ):

        status_final = "CONCLUIDA"

    else:

        status_final = (
            "AGUARDANDO_INVENTARIO"
        )

elif obra.equipe and obra.veiculo:

    if pre_apr_valida:

        status_final = status_execucao

    else:

        status_final = (
            "PENDENCIA_DOCUMENTAL"
        )

else:

    status_final = "AGUARDANDO_PROGRAMACAO"


if pendencias and status_final != "CONCLUIDA":

    if status_final == "AGUARDANDO_PROGRAMACAO":

        status_final = "PENDENTE"


obra.status = status_final


return {

    "wl": obra.wl,

    "status": status_final,

    "percentual_execucao": (
        obra.percentual_execucao
    ),

    "pendencias": pendencias,

    "inventario": resultado_inventario
}
```

# ============================================================

# INTEGRAÇÃO SIGEO HELPER

# ============================================================

#

# O SIGEOhelper_Setup.exe deve permanecer instalado

# normalmente no Windows.

#

# Esta classe prepara o EletroGestor para receber arquivos

# exportados pelo SIGEO Helper.

#

# Formatos suportados:

#

# CSV

# XLSX

# XLS

#

# O fluxo será:

#

# SIGEO HELPER

# ↓

# Exportação

# ↓

# Arquivo Excel / CSV

# ↓

# Pasta sigeo_import

# ↓

# ELETROGESTOR

# ↓

# Processamento automático

#

# ============================================================

class SIGEOHelper:

```
@staticmethod
def listar_arquivos():

    arquivos = []

    extensoes = (

        ".csv",

        ".xlsx",

        ".xls"
    )

    for arquivo in os.listdir(PASTA_SIGEO):

        caminho = os.path.join(

            PASTA_SIGEO,

            arquivo
        )

        if (

            os.path.isfile(caminho)

            and arquivo.lower().endswith(

                extensoes
            )
        ):

            arquivos.append({

                "arquivo": arquivo,

                "caminho": caminho,

                "tamanho": os.path.getsize(

                    caminho
                )
            })

    return arquivos


@staticmethod
def ler_arquivo(caminho):

    if caminho.lower().endswith(".csv"):

        try:

            df = pd.read_csv(

                caminho,

                sep=";",

                encoding="utf-8"
            )

        except Exception:

            df = pd.read_csv(

                caminho,

                encoding="latin1"
            )

    elif (

        caminho.lower().endswith(

            ".xlsx"
        )

        or

        caminho.lower().endswith(

            ".xls"
        )
    ):

        df = pd.read_excel(caminho)

    else:

        raise ValueError(

            "Formato não suportado"
        )

    df.columns = [

        str(coluna).strip().lower()

        for coluna in df.columns
    ]

    return df


@staticmethod
def normalizar_colunas(df):

    mapa = {

        "codigo": "codigo",

        "código": "codigo",

        "cod": "codigo",

        "material": "descricao",

        "descricao": "descricao",

        "descrição": "descricao",

        "quantidade": "quantidade",

        "qtd": "quantidade",

        "unidade": "unidade",

        "wl": "wl",

        "obra": "wl"
    }


    novas_colunas = {}

    for coluna in df.columns:

        coluna_limpa = (

            str(coluna)

            .strip()

            .lower()
        )

        if coluna_limpa in mapa:

            novas_colunas[coluna] = (

                mapa[coluna_limpa]
            )


    df = df.rename(

        columns=novas_colunas
    )

    return df


@staticmethod
def importar_para_materiais(

    caminho,

    db
):

    df = SIGEOHelper.ler_arquivo(

        caminho
    )

    df = SIGEOHelper.normalizar_colunas(

        df
    )


    registros_importados = 0

    erros = []


    for indice, linha in df.iterrows():

        try:

            wl = str(

                linha.get("wl", "")

            ).strip()


            codigo = str(

                linha.get("codigo", "")

            ).strip()


            descricao = str(

                linha.get(

                    "descricao",

                    ""
                )

            ).strip()


            quantidade = float(

                linha.get(

                    "quantidade",

                    0
                )

                or 0
            )


            unidade = str(

                linha.get(

                    "unidade",

                    "UN"
                )

            ).strip()


            if not wl or not codigo:

                continue


            obra = (

                db.query(Obra)

                .filter(

                    Obra.wl == wl

                )

                .first()
            )


            if not obra:

                erros.append({

                    "linha": int(

                        indice + 2
                    ),

                    "erro": (

                        f"WL {wl} não encontrada"
                    )
                })

                continue


            material = (

                db.query(MaterialObra)

                .filter(

                    MaterialObra.obra_id

                    == obra.id,

                    MaterialObra.codigo

                    == codigo

                )

                .first()
            )


            if material:

                material.quantidade_prevista += (

                    quantidade
                )

            else:

                material = MaterialObra(

                    obra_id=obra.id,

                    codigo=codigo,

                    descricao=descricao,

                    unidade=unidade,

                    quantidade_prevista=quantidade
                )

                db.add(material)


            registros_importados += 1


        except Exception as erro:

            erros.append({

                "linha": int(indice + 2),

                "erro": str(erro)
            })


    db.commit()


    return {

        "registros_importados":

            registros_importados,

        "erros": erros
    }
```

# ============================================================

# FASTAPI

# ============================================================

app = FastAPI(

```
title="EletroGestor API",

version=VERSAO,

description=(

    "Motor central de gestão de obras, "

    "materiais, viabilidade, inventário "

    "e programação."
)
```

)

# ============================================================

# CORS

# ============================================================

app.add_middleware(

```
CORSMiddleware,

allow_origins=["*"],

allow_credentials=True,

allow_methods=["*"],

allow_headers=["*"]
```

)

# ============================================================

# CRIAR BANCO

# ============================================================

Base.metadata.create_all(

```
bind=engine
```

)

# ============================================================

# ROTA PRINCIPAL

# ============================================================

@app.get("/")

def inicio():

```
return {

    "sistema": NOME_SISTEMA,

    "versao": VERSAO,

    "status": "ONLINE",

    "documentacao": "/docs"
}
```

# ============================================================

# HEALTH CHECK

# ============================================================

@app.get("/health")

def health():

```
return {

    "status": "ONLINE",

    "sistema": NOME_SISTEMA,

    "data": (

        datetime.datetime.now()
        .isoformat()
    )
}
```

# ============================================================

# CRIAR OBRA

# ============================================================

@app.post("/obras")

def criar_obra(

```
dados: ObraCreate,

db: Session = Depends(get_db)
```

):

```
obra_existente = (

    db.query(Obra)

    .filter(

        Obra.wl == dados.wl

    )

    .first()
)


if obra_existente:

    raise HTTPException(

        status_code=400,

        detail="WL já cadastrada"
    )


obra = Obra(

    wl=dados.wl,

    projeto=dados.projeto,

    circuito=dados.circuito,

    endereco=dados.endereco,

    cidade=dados.cidade,

    tipo_intervencao=(

        dados.tipo_intervencao
    )
)


db.add(obra)

db.commit()

db.refresh(obra)


return {

    "mensagem": (

        "Obra criada com sucesso"
    ),

    "id": obra.id,

    "wl": obra.wl
}
```

# ============================================================

# LISTAR OBRAS

# ============================================================

@app.get("/obras")

def listar_obras(

```
db: Session = Depends(get_db)
```

):

```
obras = (

    db.query(Obra)

    .order_by(

        Obra.criado_em.desc()
    )

    .all()
)


resultado = []


for obra in obras:

    status = calcular_status_obra(

        obra,

        db
    )


    resultado.append({

        "id": obra.id,

        "wl": obra.wl,

        "projeto": obra.projeto,

        "cidade": obra.cidade,

        "status": status["status"],

        "percentual_execucao":

            obra.percentual_execucao,

        "pendencias":

            status["pendencias"]
    })


db.commit()


return resultado
```

# ============================================================

# BUSCAR OBRA

# ============================================================

@app.get("/obras/{obra_id}")

def buscar_obra(

```
obra_id: int,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


status = calcular_status_obra(

    obra,

    db
)


return {

    "id": obra.id,

    "wl": obra.wl,

    "projeto": obra.projeto,

    "circuito": obra.circuito,

    "endereco": obra.endereco,

    "cidade": obra.cidade,

    "tipo_intervencao":

        obra.tipo_intervencao,

    "status":

        status["status"],

    "percentual_execucao":

        obra.percentual_execucao,

    "pendencias":

        status["pendencias"]
}
```

# ============================================================

# ATUALIZAR PERCENTUAL

# ============================================================

@app.put("/obras/{obra_id}/execucao")

def atualizar_execucao(

```
obra_id: int,

percentual: float,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


if percentual < 0 or percentual > 100:

    raise HTTPException(

        status_code=400,

        detail=(

            "Percentual deve estar "

            "entre 0 e 100"
        )
    )


obra.percentual_execucao = percentual


db.commit()


status = calcular_status_obra(

    obra,

    db
)


db.commit()


return status
```

# ============================================================

# CRIAR VIABILIDADE

# ============================================================

@app.post("/obras/{obra_id}/viabilidade")

def criar_viabilidade(

```
obra_id: int,

dados: ViabilidadeCreate,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


if obra.viabilidade:

    raise HTTPException(

        status_code=400,

        detail=(

            "Viabilidade já cadastrada"
        )
    )


viabilidade = Viabilidade(

    obra_id=obra_id,

    responsavel=dados.responsavel,

    data_inspecao=dados.data_inspecao,

    status=dados.status,

    vistoria_manobra_realizada=(

        dados.vistoria_manobra_realizada
    ),

    data_vistoria_manobra=(

        dados.data_vistoria_manobra
    ),

    observacoes_manobra=(

        dados.observacoes_manobra
    ),

    observacoes_gerais=(

        dados.observacoes_gerais
    )
)


db.add(viabilidade)

db.commit()


return {

    "mensagem":

        "Viabilidade cadastrada",

    "obra_id": obra_id
}
```

# ============================================================

# CONSULTAR VIABILIDADE

# ============================================================

@app.get("/obras/{obra_id}/viabilidade")

def consultar_viabilidade(

```
obra_id: int,

db: Session = Depends(get_db)
```

):

```
viabilidade = (

    db.query(Viabilidade)

    .filter(

        Viabilidade.obra_id == obra_id

    )

    .first()
)


if not viabilidade:

    raise HTTPException(

        status_code=404,

        detail=(

            "Viabilidade não encontrada"
        )
    )


return {

    "id": viabilidade.id,

    "responsavel":

        viabilidade.responsavel,

    "data_inspecao":

        viabilidade.data_inspecao,

    "status":

        viabilidade.status,

    "vistoria_manobra":

        viabilidade.vistoria_manobra_realizada,

    "observacoes":

        viabilidade.observacoes_gerais
}
```

# ============================================================

# CRIAR PRE APR

# ============================================================

@app.post("/obras/{obra_id}/pre-apr")

def criar_pre_apr(

```
obra_id: int,

dados: PreAPRCreate,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


resultado = verificar_pre_apr(

    dados.data_elaboracao,

    dados.validade_dias
)


pre_apr = PreAPR(

    obra_id=obra_id,

    responsavel=dados.responsavel,

    data_elaboracao=(

        dados.data_elaboracao
    ),

    validade_dias=(

        dados.validade_dias
    ),

    status=resultado["status"],

    observacoes=dados.observacoes
)


db.add(pre_apr)

db.commit()


return resultado
```

# ============================================================

# CONSULTAR PRE APR

# ============================================================

@app.get("/obras/{obra_id}/pre-apr")

def consultar_pre_apr(

```
obra_id: int,

db: Session = Depends(get_db)
```

):

```
pre_aprs = (

    db.query(PreAPR)

    .filter(

        PreAPR.obra_id == obra_id

    )

    .order_by(

        PreAPR.data_elaboracao.desc()
    )

    .all()
)


resultado = []


for pre_apr in pre_aprs:

    status = verificar_pre_apr(

        pre_apr.data_elaboracao,

        pre_apr.validade_dias
    )


    pre_apr.status = (

        status["status"]
    )


    resultado.append({

        "id": pre_apr.id,

        "responsavel":

            pre_apr.responsavel,

        **status
    })


db.commit()


return resultado
```

# ============================================================

# ADICIONAR MATERIAL

# ============================================================

@app.post("/obras/{obra_id}/materiais")

def adicionar_material(

```
obra_id: int,

dados: MaterialCreate,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


material = MaterialObra(

    obra_id=obra_id,

    codigo=dados.codigo,

    descricao=dados.descricao,

    aplicacao=dados.aplicacao,

    unidade=dados.unidade,

    quantidade_prevista=(

        dados.quantidade_prevista
    ),

    quantidade_retirada=(

        dados.quantidade_retirada
    ),

    quantidade_instalada=(

        dados.quantidade_instalada
    ),

    quantidade_devolvida=(

        dados.quantidade_devolvida
    )
)


db.add(material)

db.commit()

db.refresh(material)


return reconciliar_material(

    material
)
```

# ============================================================

# LISTAR MATERIAIS

# ============================================================

@app.get("/obras/{obra_id}/materiais")

def listar_materiais(

```
obra_id: int,

consolidado: bool = False,

db: Session = Depends(get_db)
```

):

```
materiais = (

    db.query(MaterialObra)

    .filter(

        MaterialObra.obra_id == obra_id

    )

    .all()
)


if consolidado:

    return consolidar_materiais(

        materiais
    )


return [

    reconciliar_material(material)

    for material in materiais
]
```

# ============================================================

# RETIRAR MATERIAL

# ============================================================

@app.post("/materiais/{material_id}/retirar")

def retirar_material(

```
material_id: int,

dados: MovimentoMaterial,

db: Session = Depends(get_db)
```

):

```
material = db.get(

    MaterialObra,

    material_id
)


if not material:

    raise HTTPException(

        status_code=404,

        detail="Material não encontrado"
    )


material.quantidade_retirada += (

    dados.quantidade
)


db.commit()


return reconciliar_material(

    material
)
```

# ============================================================

# INSTALAR MATERIAL

# ============================================================

@app.post("/materiais/{material_id}/instalar")

def instalar_material(

```
material_id: int,

dados: MovimentoMaterial,

db: Session = Depends(get_db)
```

):

```
material = db.get(

    MaterialObra,

    material_id
)


if not material:

    raise HTTPException(

        status_code=404,

        detail="Material não encontrado"
    )


material.quantidade_instalada += (

    dados.quantidade
)


db.commit()


return reconciliar_material(

    material
)
```

# ============================================================

# DEVOLVER MATERIAL

# ============================================================

@app.post("/materiais/{material_id}/devolver")

def devolver_material(

```
material_id: int,

dados: MovimentoMaterial,

db: Session = Depends(get_db)
```

):

```
material = db.get(

    MaterialObra,

    material_id
)


if not material:

    raise HTTPException(

        status_code=404,

        detail="Material não encontrado"
    )


material.quantidade_devolvida += (

    dados.quantidade
)


db.commit()


return reconciliar_material(

    material
)
```

# ============================================================

# CRIAR EQUIPE

# ============================================================

@app.post("/equipes")

def criar_equipe(

```
dados: EquipeCreate,

db: Session = Depends(get_db)
```

):

```
existe = (

    db.query(Equipe)

    .filter(

        Equipe.codigo == dados.codigo

    )

    .first()
)


if existe:

    raise HTTPException(

        status_code=400,

        detail="Código de equipe já existe"
    )


equipe = Equipe(

    nome=dados.nome,

    codigo=dados.codigo,

    disponivel=dados.disponivel,

    certificacao_valida=(

        dados.certificacao_valida
    ),

    especialidade=(

        dados.especialidade
    ),

    cidade_base=(

        dados.cidade_base
    )
)


db.add(equipe)

db.commit()

db.refresh(equipe)


return {

    "id": equipe.id,

    "nome": equipe.nome,

    "codigo": equipe.codigo
}
```

# ============================================================

# LISTAR EQUIPES

# ============================================================

@app.get("/equipes")

def listar_equipes(

```
db: Session = Depends(get_db)
```

):

```
equipes = db.query(

    Equipe

).all()


return [

    {

        "id": equipe.id,

        "codigo": equipe.codigo,

        "nome": equipe.nome,

        "disponivel":

            equipe.disponivel,

        "certificacao_valida":

            equipe.certificacao_valida,

        "especialidade":

            equipe.especialidade,

        "cidade_base":

            equipe.cidade_base
    }

    for equipe in equipes
]
```

# ============================================================

# RECOMENDAR EQUIPE

# ============================================================

@app.get("/obras/{obra_id}/recomendar-equipe")

def recomendar_equipe_para_obra(

```
obra_id: int,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


equipes = (

    db.query(Equipe)

    .filter(

        Equipe.disponivel == True

    )

    .all()
)


recomendacoes = recomendar_equipe(

    obra,

    equipes
)


return {

    "obra": obra.wl,

    "recomendacoes":

        recomendacoes
}
```

# ============================================================

# CRIAR VEICULO

# ============================================================

@app.post("/veiculos")

def criar_veiculo(

```
dados: VeiculoCreate,

db: Session = Depends(get_db)
```

):

```
existe = (

    db.query(Veiculo)

    .filter(

        Veiculo.placa == dados.placa

    )

    .first()
)


if existe:

    raise HTTPException(

        status_code=400,

        detail="Veículo já cadastrado"
    )


veiculo = Veiculo(

    placa=dados.placa,

    tipo=dados.tipo,

    disponivel=dados.disponivel
)


db.add(veiculo)

db.commit()

db.refresh(veiculo)


return {

    "id": veiculo.id,

    "placa": veiculo.placa
}
```

# ============================================================

# LISTAR VEICULOS

# ============================================================

@app.get("/veiculos")

def listar_veiculos(

```
db: Session = Depends(get_db)
```

):

```
veiculos = db.query(

    Veiculo

).all()


return [

    {

        "id": veiculo.id,

        "placa": veiculo.placa,

        "tipo": veiculo.tipo,

        "disponivel":

            veiculo.disponivel
    }

    for veiculo in veiculos
]
```

# ============================================================

# PROGRAMAR OBRA

# ============================================================

@app.post("/obras/{obra_id}/programar")

def programar_obra(

```
obra_id: int,

dados: ProgramacaoCreate,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


equipe = db.get(

    Equipe,

    dados.equipe_id
)


if not equipe:

    raise HTTPException(

        status_code=404,

        detail="Equipe não encontrada"
    )


veiculo = db.get(

    Veiculo,

    dados.veiculo_id
)


if not veiculo:

    raise HTTPException(

        status_code=404,

        detail="Veículo não encontrado"
    )


if not equipe.disponivel:

    raise HTTPException(

        status_code=400,

        detail="Equipe indisponível"
    )


if not equipe.certificacao_valida:

    raise HTTPException(

        status_code=400,

        detail=(

            "Equipe possui "

            "certificação inválida"
        )
    )


if not veiculo.disponivel:

    raise HTTPException(

        status_code=400,

        detail="Veículo indisponível"
    )


obra.equipe = equipe.codigo

obra.veiculo = veiculo.placa

obra.data_programada = (

    dados.data_programada
)


db.commit()


status = calcular_status_obra(

    obra,

    db
)


db.commit()


return {

    "mensagem":

        "Obra programada com sucesso",

    "programacao": {

        "wl": obra.wl,

        "equipe": equipe.codigo,

        "veiculo": veiculo.placa,

        "data":

            dados.data_programada
    },

    "status": status
}
```

# ============================================================

# CRIAR INVENTARIO

# ============================================================

@app.post("/obras/{obra_id}/inventario")

def criar_inventario(

```
obra_id: int,

dados: InventarioCreate,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


inventario = Inventario(

    obra_id=obra_id,

    responsavel=dados.responsavel,

    foto_antes=dados.foto_antes,

    foto_depois=dados.foto_depois,

    observacoes=dados.observacoes
)


resultado = verificar_inventario(

    [inventario]
)


inventario.status = (

    resultado["status"]
)


db.add(inventario)

db.commit()


return resultado
```

# ============================================================

# FATURAMENTO

# ============================================================

@app.post("/obras/{obra_id}/faturamento")

def criar_faturamento(

```
obra_id: int,

dados: FaturamentoCreate,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


faturamento = Faturamento(

    obra_id=obra_id,

    valor_previsto=(

        dados.valor_previsto
    ),

    valor_executado=(

        dados.valor_executado
    ),

    valor_faturado=(

        dados.valor_faturado
    ),

    status=dados.status
)


db.add(faturamento)

db.commit()


return {

    "mensagem":

        "Faturamento cadastrado",

    "obra_id": obra_id
}
```

# ============================================================

# IMPORTAR EXCEL NORMAL

# ============================================================

@app.post("/importar-excel")

async def importar_excel(

```
arquivo: UploadFile = File(...),

db: Session = Depends(get_db)
```

):

```
extensao = os.path.splitext(

    arquivo.filename

)[1].lower()


if extensao not in [

    ".xlsx",

    ".xls",

    ".csv"
]:

    raise HTTPException(

        status_code=400,

        detail=(

            "Formato inválido. "

            "Envie XLSX, XLS ou CSV."
        )
    )


caminho = os.path.join(

    PASTA_UPLOADS,

    arquivo.filename
)


with open(

    caminho,

    "wb"

) as buffer:

    shutil.copyfileobj(

        arquivo.file,

        buffer
    )


resultado = (

    SIGEOHelper.importar_para_materiais(

        caminho,

        db
    )
)


return {

    "arquivo":

        arquivo.filename,

    **resultado
}
```

# ============================================================

# SIGEO HELPER - STATUS

# ============================================================

@app.get("/sigeo/status")

def sigeo_status():

```
arquivos = (

    SIGEOHelper.listar_arquivos()
)


return {

    "integracao":

        "SIGEO HELPER PREPARADA",

    "pasta":

        os.path.abspath(

            PASTA_SIGEO
        ),

    "arquivos":

        arquivos
}
```

# ============================================================

# SIGEO HELPER - LISTAR ARQUIVOS

# ============================================================

@app.get("/sigeo/arquivos")

def sigeo_arquivos():

```
return (

    SIGEOHelper.listar_arquivos()
)
```

# ============================================================

# SIGEO HELPER - IMPORTAR ARQUIVO

# ============================================================

@app.post("/sigeo/importar/{nome_arquivo}")

def sigeo_importar(

```
nome_arquivo: str,

db: Session = Depends(get_db)
```

):

```
caminho = os.path.join(

    PASTA_SIGEO,

    nome_arquivo
)


if not os.path.exists(caminho):

    raise HTTPException(

        status_code=404,

        detail=(

            "Arquivo não encontrado "

            "na pasta SIGEO"
        )
    )


try:

    resultado = (

        SIGEOHelper.importar_para_materiais(

            caminho,

            db
        )
    )


    return {

        "arquivo":

            nome_arquivo,

        **resultado
    }


except Exception as erro:

    raise HTTPException(

        status_code=500,

        detail=str(erro)
    )
```

# ============================================================

# DASHBOARD

# ============================================================

@app.get("/dashboard")

def dashboard(

```
db: Session = Depends(get_db)
```

):

```
obras = db.query(

    Obra

).all()


total_obras = len(obras)

concluidas = 0

pendentes = 0

em_execucao = 0

aguardando_inventario = 0


for obra in obras:

    status = calcular_status_obra(

        obra,

        db
    )


    if status["status"] == "CONCLUIDA":

        concluidas += 1


    elif status["status"] == "EM_EXECUCAO":

        em_execucao += 1


    elif (

        status["status"]

        == "AGUARDANDO_INVENTARIO"
    ):

        aguardando_inventario += 1


    else:

        pendentes += 1


materiais = db.query(

    MaterialObra

).all()


saldo_total = sum(

    (

        material.quantidade_retirada

        - material.quantidade_instalada

        - material.quantidade_devolvida
    )

    for material in materiais
)


db.commit()


return {

    "obras": {

        "total": total_obras,

        "concluidas": concluidas,

        "pendentes": pendentes,

        "em_execucao":

            em_execucao,

        "aguardando_inventario":

            aguardando_inventario
    },

    "materiais": {

        "total_itens":

            len(materiais),

        "saldo_total":

            saldo_total
    }
}
```

# ============================================================

# STATUS COMPLETO DA OBRA

# ============================================================

@app.get("/obras/{obra_id}/status-completo")

def status_completo(

```
obra_id: int,

db: Session = Depends(get_db)
```

):

```
obra = db.get(

    Obra,

    obra_id
)


if not obra:

    raise HTTPException(

        status_code=404,

        detail="Obra não encontrada"
    )


resultado = calcular_status_obra(

    obra,

    db
)


materiais = [

    reconciliar_material(material)

    for material in obra.materiais
]


db.commit()


return {

    "obra": {

        "id": obra.id,

        "wl": obra.wl,

        "projeto": obra.projeto
    },

    "status": resultado,

    "materiais": materiais,

    "programacao": {

        "equipe": obra.equipe,

        "veiculo": obra.veiculo,

        "data_programada":

            obra.data_programada
    }
}
```

# ============================================================

# EXECUÇÃO DIRETA

# ============================================================

if **name** == "**main**":

```
import uvicorn


uvicorn.run(

    app,

    host="0.0.0.0",

    port=8000,

    reload=True
)
```


IndentationError: expected an indented block after function definition on line 150 (2973996777.py, line 152)

In [ ]:
#logica pythoon gestão de frota
from datetime import datetime

# Registros do sistema (exemplo de simulação)
solicitacoes = [
    # [0] Motorista, [1] Cat CNH, [2] Venc. CNH (ano, mes, dia), [3] Termo Assinado, [4] Placa, [5] Tipo Veiculo, [6] CRLV OK, [7] Checklist OK, [8] Finalidade, [9] OS TEES, [10] OS ENEL, [11] Liberacao Frota, [12] Ocupantes, [13] Vistoria OK
    ["João da Silva", "D", [2027, 12, 31], True, "ABC-1234", "Pesado", True, True, "Inventario", True, True, True, 1, True],
    ["Maria Souza", "C", [2027, 5, 20], True, "DEF-5678", "Pesado", True, True, "Operacao", True, False, False, 1, True],
    ["Carlos Lima", "B", [2027, 8, 15], True, "GHI-9012", "Leve", True, True, "Inventario", False, False, False, 2, True],
    ["Ana Costa", "E", [2027, 10, 10], True, "JKL-3456", "Pesado", True, True, "Operacao", True, True, False, 1, True]
]

data_hoje = [2026, 8, 28]
indice = 0

# Processa cada solicitação na fila usando WHILE
while indice < len(solicitacoes):
    veiculo_atual = solicitacoes[indice]

    motorista_nome = veiculo_atual[0]
    cnh_categoria = veiculo_atual[1]
    cnh_vencimento = veiculo_atual[2]
    termo_assinado = veiculo_atual[3]
    placa = veiculo_atual[4]
    tipo_veiculo = veiculo_atual[5]
    crlv_ok = veiculo_atual[6]
    checklist_ok = veiculo_atual[7]
    finalidade = veiculo_atual[8]
    os_tees = veiculo_atual[9]
    os_enel = veiculo_atual[10]
    liberacao_frota = veiculo_atual[11]
    ocupantes = veiculo_atual[12]
    vistoria_ok = veiculo_atual[13]

    bloqueios = []

    # 1. Validações Gerais do Condutor e Veículo
    if not termo_assinado:
        bloqueios.append("Sem Termo de Responsabilidade assinado.")

    if cnh_vencimento < data_hoje:
        bloqueios.append("CNH Vencida.")

    if not crlv_ok:
        bloqueios.append("CRLV irregular.")

    if not checklist_ok:
        bloqueios.append("Checklist diário não aprovado.")

    if not vistoria_ok:
        bloqueios.append("Vistoria conjunta da portaria não realizada].")

    # 2. Regras por Tipo de Veículo
    if tipo_veiculo == "Leve":
        if cnh_categoria not in ["B", "C", "D", "E"]:
            bloqueios.append("CNH incompatível para veículo leve.")

    elif tipo_veiculo == "Pesado":
        if cnh_categoria not in ["C", "D", "E"]:
            bloqueios.append("CNH incompatível para veículo pesado.")

        # Regra do Inventário
        if finalidade == "Inventario":
            bloqueios.append("BLOQUeIO CRÍTICO: Veículo pesado ptoibido para inventario.")

        # Regra de Operação
        elif finalidade == "Operacao":
            if not os_tees:
                bloqueios.append("Falta O.S. da TEES assinada pelo gesTor.")
            if not os_enel:
                bloqueios.append("Falta O.S. ou PI da ENEL vinculada.")

        # Regra de Manutenção
        elif finalidade == "Manutencao":
            if not liberacao_frota:
                bloqueios.append("Falta liberação da Equipe de Frota.")
            if not os_tees:
                bloqueios.append("Falta O.S. da TEES assinada pelo ADM.")
            if ocupantes > 2:
                bloqueios.append("Proibido equipe na manutenção (permitido apenas motorista e auxiliar).")

    # 3. Decisão da Saída
    print(f"\n Analisando: {placa} ({motorista_nome}) ")
    if len(bloqueios) == 0:
        print("STATUS: SAÍDA LIBERADA[cite: 2]")
    else:
        print("STATUS: SAÍDA BLOQUEADA[cite: 2]")
        sub_indice = 0
        while sub_indice < len(bloqueios):
            print(f" - Motivo: {bloqueios[sub_indice]}")
            sub_indice += 1

    indice += 1


 Analisando: ABC-1234 (João da Silva) 
STATUS: SAÍDA BLOQUEADA[cite: 2]
 - Motivo: BLOQUeIO CRÍTICO: Veículo pesado ptoibido para inventario.

 Analisando: DEF-5678 (Maria Souza) 
STATUS: SAÍDA BLOQUEADA[cite: 2]
 - Motivo: Falta O.S. ou PI da ENEL vinculada.

 Analisando: GHI-9012 (Carlos Lima) 
STATUS: SAÍDA LIBERADA[cite: 2]

 Analisando: JKL-3456 (Ana Costa) 
STATUS: SAÍDA LIBERADA[cite: 2]


Core + Autenticação

In [ ]:
# ELETROGESTOR 2.0 - PARTE 1/N: CORE + AUTENTICAÇÃO
# (cole esta célula inteira no Google Colab e rode)

!pip install sqlalchemy pyotp -q

import os
import uuid
import hashlib
import secrets
import pyotp
from datetime import datetime, timedelta
from enum import Enum

from sqlalchemy import (
    create_engine, Column, String, Integer, Boolean, DateTime,
    ForeignKey, Text, Enum as SAEnum
)
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

# ------------------------------------------------------------
# CORE > CONFIGURAÇÃO
# ------------------------------------------------------------

class Config:
    DB_URL = os.getenv("ELETROGESTOR_DB_URL", "sqlite:///eletrogestor.db")
    SECRET_KEY = os.getenv("ELETROGESTOR_SECRET", secrets.token_hex(32))
    SESSION_TIMEOUT_MIN = int(os.getenv("SESSION_TIMEOUT_MIN", "60"))
    SENHA_MIN_LEN = 8
    TWOFA_ISSUER = "EletroGestor"
    CERTIFICACAO_VALIDADE_DIAS = 45  # regra: validade de 45 dias (SEGURANÇA)


# ------------------------------------------------------------
# CORE > BANCO DE DADOS
# ------------------------------------------------------------

Base = declarative_base()
engine = create_engine(Config.DB_URL, echo=False)
SessionLocal = sessionmaker(bind=engine)


def get_session():
    return SessionLocal()


def init_db():
    Base.metadata.create_all(engine)


# ------------------------------------------------------------
# USUÁRIOS > PERFIS (RBAC)
# ------------------------------------------------------------

class PerfilUsuario(str, Enum):
    SUPERVISOR = "supervisor"
    ALMOXARIFADO = "almoxarifado"
    OPERACIONAL = "operacional"
    ADMINISTRATIVO = "administrativo"
    DESENVOLVEDOR = "desenvolvedor"
    TESTER = "tester"
    ADMINISTRADOR = "administrador"


# ------------------------------------------------------------
# USUÁRIOS > CADASTRO
# ------------------------------------------------------------

class Usuario(Base):
    __tablename__ = "usuarios"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    nome = Column(String(150), nullable=False)
    email = Column(String(150), unique=True, nullable=False)
    senha_hash = Column(String(255), nullable=False)
    salt = Column(String(64), nullable=False)
    perfil = Column(SAEnum(PerfilUsuario), nullable=False, default=PerfilUsuario.OPERACIONAL)
    ativo = Column(Boolean, default=True)
    totp_secret = Column(String(64), nullable=True)  # 2FA
    twofa_ativo = Column(Boolean, default=False)
    criado_em = Column(DateTime, default=datetime.utcnow)
    ultimo_login = Column(DateTime, nullable=True)

    sessoes = relationship("Sessao", back_populates="usuario")
    logs = relationship("LogAuditoria", back_populates="usuario")


# ------------------------------------------------------------
# CORE > SEGURANÇA (hash de senha)
# ------------------------------------------------------------

class SegurancaSenha:
    @staticmethod
    def gerar_salt() -> str:
        return secrets.token_hex(16)

    @staticmethod
    def hash_senha(senha: str, salt: str) -> str:
        return hashlib.sha256((salt + senha).encode()).hexdigest()

    @staticmethod
    def validar_senha(senha: str) -> bool:
        return len(senha) >= Config.SENHA_MIN_LEN

    @classmethod
    def criar_hash(cls, senha: str):
        salt = cls.gerar_salt()
        return cls.hash_senha(senha, salt), salt

    @classmethod
    def verificar(cls, senha: str, salt: str, hash_armazenado: str) -> bool:
        return cls.hash_senha(senha, salt) == hash_armazenado


# ------------------------------------------------------------
# AUTENTICAÇÃO > SESSÃO
# ------------------------------------------------------------

class Sessao(Base):
    __tablename__ = "sessoes"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    usuario_id = Column(String(36), ForeignKey("usuarios.id"), nullable=False)
    token = Column(String(255), unique=True, nullable=False)
    criado_em = Column(DateTime, default=datetime.utcnow)
    expira_em = Column(DateTime, nullable=False)
    ativa = Column(Boolean, default=True)

    usuario = relationship("Usuario", back_populates="sessoes")


# ------------------------------------------------------------
# CORE > AUDITORIA / LOGS
# ------------------------------------------------------------

class LogAuditoria(Base):
    __tablename__ = "logs_auditoria"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    usuario_id = Column(String(36), ForeignKey("usuarios.id"), nullable=True)
    acao = Column(String(150), nullable=False)
    detalhes = Column(Text, nullable=True)
    ip = Column(String(45), nullable=True)
    criado_em = Column(DateTime, default=datetime.utcnow)

    usuario = relationship("Usuario", back_populates="logs")


def registrar_log(session, usuario_id, acao, detalhes=None, ip=None):
    log = LogAuditoria(usuario_id=usuario_id, acao=acao, detalhes=detalhes, ip=ip)
    session.add(log)
    session.commit()
    return log


# ------------------------------------------------------------
# AUTENTICAÇÃO > SERVIÇO (Login, Logout, Recuperar Senha, 2FA)
# ------------------------------------------------------------

class AuthService:
    def __init__(self, session):
        self.session = session

    def cadastrar_usuario(self, nome, email, senha, perfil=PerfilUsuario.OPERACIONAL):
        if not SegurancaSenha.validar_senha(senha):
            raise ValueError(f"Senha deve ter no mínimo {Config.SENHA_MIN_LEN} caracteres")

        hash_, salt = SegurancaSenha.criar_hash(senha)
        usuario = Usuario(
            nome=nome, email=email, senha_hash=hash_, salt=salt, perfil=perfil
        )
        self.session.add(usuario)
        self.session.commit()
        registrar_log(self.session, usuario.id, "CADASTRO_USUARIO", f"perfil={perfil}")
        return usuario

    def login(self, email, senha, codigo_2fa=None, ip=None):
        usuario = self.session.query(Usuario).filter_by(email=email, ativo=True).first()
        if not usuario:
            raise ValueError("Usuário não encontrado ou inativo")

        if not SegurancaSenha.verificar(senha, usuario.salt, usuario.senha_hash):
            registrar_log(self.session, usuario.id, "LOGIN_FALHA", "senha incorreta", ip)
            raise ValueError("Credenciais inválidas")

        if usuario.twofa_ativo:
            if not codigo_2fa or not self.validar_2fa(usuario, codigo_2fa):
                registrar_log(self.session, usuario.id, "LOGIN_FALHA", "2FA inválido", ip)
                raise ValueError("Código 2FA inválido")

        token = secrets.token_urlsafe(48)
        expira = datetime.utcnow() + timedelta(minutes=Config.SESSION_TIMEOUT_MIN)
        sessao = Sessao(usuario_id=usuario.id, token=token, expira_em=expira)
        self.session.add(sessao)
        usuario.ultimo_login = datetime.utcnow()
        self.session.commit()

        registrar_log(self.session, usuario.id, "LOGIN_SUCESSO", ip=ip)
        return sessao

    def logout(self, token):
        sessao = self.session.query(Sessao).filter_by(token=token, ativa=True).first()
        if sessao:
            sessao.ativa = False
            self.session.commit()
            registrar_log(self.session, sessao.usuario_id, "LOGOUT")
        return True

    def validar_sessao(self, token):
        sessao = self.session.query(Sessao).filter_by(token=token, ativa=True).first()
        if not sessao or sessao.expira_em < datetime.utcnow():
            return None
        return sessao

    def gerar_token_recuperacao(self, email):
        usuario = self.session.query(Usuario).filter_by(email=email).first()
        if not usuario:
            raise ValueError("Usuário não encontrado")
        token = secrets.token_urlsafe(32)
        registrar_log(self.session, usuario.id, "SOLICITA_RECUPERACAO_SENHA")
        return token

    def redefinir_senha(self, usuario_id, nova_senha):
        if not SegurancaSenha.validar_senha(nova_senha):
            raise ValueError(f"Senha deve ter no mínimo {Config.SENHA_MIN_LEN} caracteres")
        usuario = self.session.query(Usuario).get(usuario_id)
        usuario.senha_hash, usuario.salt = SegurancaSenha.criar_hash(nova_senha)
        self.session.commit()
        registrar_log(self.session, usuario_id, "SENHA_REDEFINIDA")
        return True

    def ativar_2fa(self, usuario_id):
        usuario = self.session.query(Usuario).get(usuario_id)
        usuario.totp_secret = pyotp.random_base32()
        usuario.twofa_ativo = True
        self.session.commit()
        uri = pyotp.totp.TOTP(usuario.totp_secret).provisioning_uri(
            name=usuario.email, issuer_name=Config.TWOFA_ISSUER
        )
        return usuario.totp_secret, uri

    def validar_2fa(self, usuario, codigo):
        if not usuario.totp_secret:
            return False
        totp = pyotp.totp.TOTP(usuario.totp_secret)
        return totp.verify(codigo)


# ------------------------------------------------------------
# Exemplo de uso rápido
# ------------------------------------------------------------

init_db()
db = get_session()
auth = AuthService(db)

novo = auth.cadastrar_usuario(
    "João Silva", "joao@eletrogestor.com", "SenhaForte123",
    perfil=PerfilUsuario.SUPERVISOR
)
print("Usuário criado:", novo.id)

sessao = auth.login("joao@eletrogestor.com", "SenhaForte123")
print("Sessão criada, token:", sessao.token)

Usuário criado: 52246c36-1135-4b72-a550-c63397fe18de
Sessão criada, token: hvY0xrhLlnANBfhHTeXUITl6h9Qu4elqcf_0C2IVTRTNYPbBRJUsc-Qn2CWQeCuA


/tmp/ipykernel_3895/2895038450.py:191: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  expira = datetime.utcnow() + timedelta(minutes=Config.SESSION_TIMEOUT_MIN)
/tmp/ipykernel_3895/2895038450.py:194: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  usuario.ultimo_login = datetime.utcnow()


SEGURANÇA (Localização, PI, SIGEO, Responsáveis, Equipes, Contatos, APR, Certificações, Validade 45 dias)

In [ ]:
# ============================================================
# ELETROGESTOR 2.0 - PARTE 2/N: SEGURANÇA
# (rode a Parte 1 antes, na mesma sessão do Colab)
# ============================================================

from sqlalchemy import Float, Table

# ------------------------------------------------------------
# SEGURANÇA > EQUIPES / RESPONSÁVEIS / CONTATOS
# ------------------------------------------------------------

# Tabela associativa: membros de uma equipe
equipe_membros = Table(
    "equipe_membros", Base.metadata,
    Column("equipe_id", String(36), ForeignKey("equipes.id"), primary_key=True),
    Column("usuario_id", String(36), ForeignKey("usuarios.id"), primary_key=True),
)


class Equipe(Base):
    __tablename__ = "equipes"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    nome = Column(String(150), nullable=False)
    responsavel_id = Column(String(36), ForeignKey("usuarios.id"), nullable=False)
    ativa = Column(Boolean, default=True)
    criado_em = Column(DateTime, default=datetime.utcnow)

    responsavel = relationship("Usuario", foreign_keys=[responsavel_id])
    membros = relationship("Usuario", secondary=equipe_membros)
    contatos = relationship("Contato", back_populates="equipe")


class Contato(Base):
    __tablename__ = "contatos"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    equipe_id = Column(String(36), ForeignKey("equipes.id"), nullable=False)
    nome = Column(String(150), nullable=False)
    telefone = Column(String(30), nullable=True)
    email = Column(String(150), nullable=True)
    tipo = Column(String(50), default="operacional")  # operacional, emergência, cliente...

    equipe = relationship("Equipe", back_populates="contatos")


# ------------------------------------------------------------
# SEGURANÇA > CERTIFICAÇÕES (Validade 45 dias)
# ------------------------------------------------------------

class Certificacao(Base):
    __tablename__ = "certificacoes"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    usuario_id = Column(String(36), ForeignKey("usuarios.id"), nullable=False)
    tipo = Column(String(100), nullable=False)  # ex: NR-10, NR-35, Trabalho em Altura...
    emitido_em = Column(DateTime, default=datetime.utcnow)
    validade_dias = Column(Integer, default=Config.CERTIFICACAO_VALIDADE_DIAS)

    usuario = relationship("Usuario")

    @property
    def data_expiracao(self):
        return self.emitido_em + timedelta(days=self.validade_dias)

    @property
    def valida(self):
        return datetime.utcnow() <= self.data_expiracao

    @property
    def dias_restantes(self):
        return (self.data_expiracao - datetime.utcnow()).days


# ------------------------------------------------------------
# SEGURANÇA > LOCALIZAÇÃO / PI / SIGEO
# ------------------------------------------------------------

class Localizacao(Base):
    __tablename__ = "localizacoes"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    descricao = Column(String(255), nullable=False)
    latitude = Column(Float, nullable=False)
    longitude = Column(Float, nullable=False)
    pi = Column(String(100), nullable=True)       # Ponto de Instalação/Interconexão
    sigeo = Column(String(100), nullable=True)     # Código/Registro no sistema SIGEO
    criado_em = Column(DateTime, default=datetime.utcnow)


# ------------------------------------------------------------
# SEGURANÇA > APR (Análise Preliminar de Risco)
# ------------------------------------------------------------

class APR(Base):
    __tablename__ = "aprs"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    equipe_id = Column(String(36), ForeignKey("equipes.id"), nullable=False)
    localizacao_id = Column(String(36), ForeignKey("localizacoes.id"), nullable=False)
    riscos_identificados = Column(Text, nullable=True)
    medidas_controle = Column(Text, nullable=True)
    aprovado = Column(Boolean, default=False)
    aprovado_por = Column(String(36), ForeignKey("usuarios.id"), nullable=True)
    criado_em = Column(DateTime, default=datetime.utcnow)

    equipe = relationship("Equipe")
    localizacao = relationship("Localizacao")


# ------------------------------------------------------------
# SEGURANÇA > SERVIÇO
# ------------------------------------------------------------

class SegurancaService:
    def __init__(self, session):
        self.session = session

    # ---- Equipes ----
    def criar_equipe(self, nome, responsavel_id, membros_ids=None):
        equipe = Equipe(nome=nome, responsavel_id=responsavel_id)
        if membros_ids:
            membros = self.session.query(Usuario).filter(Usuario.id.in_(membros_ids)).all()
            equipe.membros = membros
        self.session.add(equipe)
        self.session.commit()
        registrar_log(self.session, responsavel_id, "CRIAR_EQUIPE", nome)
        return equipe

    def adicionar_contato(self, equipe_id, nome, telefone=None, email=None, tipo="operacional"):
        contato = Contato(equipe_id=equipe_id, nome=nome, telefone=telefone, email=email, tipo=tipo)
        self.session.add(contato)
        self.session.commit()
        return contato

    # ---- Certificações ----
    def emitir_certificacao(self, usuario_id, tipo, validade_dias=None):
        cert = Certificacao(
            usuario_id=usuario_id,
            tipo=tipo,
            validade_dias=validade_dias or Config.CERTIFICACAO_VALIDADE_DIAS,
        )
        self.session.add(cert)
        self.session.commit()
        return cert

    def certificacoes_vencendo(self, dias_alerta=5):
        certs = self.session.query(Certificacao).all()
        return [c for c in certs if c.valida and c.dias_restantes <= dias_alerta]

    def certificacoes_vencidas(self):
        certs = self.session.query(Certificacao).all()
        return [c for c in certs if not c.valida]

    # ---- Validação de equipe (pré-requisito p/ liberar a obra) ----
    def validar_equipe(self, equipe_id):
        """Verifica se todos os membros têm certificações válidas."""
        equipe = self.session.query(Equipe).get(equipe_id)
        if not equipe or not equipe.ativa:
            return False, "Equipe inexistente ou inativa"

        pendencias = []
        for membro in equipe.membros:
            certs = self.session.query(Certificacao).filter_by(usuario_id=membro.id).all()
            if not certs or not any(c.valida for c in certs):
                pendencias.append(membro.nome)

        if pendencias:
            return False, f"Membros com certificação vencida/ausente: {', '.join(pendencias)}"
        return True, "Equipe validada"

    # ---- Localização + Pré-APR ----
    def cadastrar_localizacao(self, descricao, latitude, longitude, pi=None, sigeo=None):
        loc = Localizacao(descricao=descricao, latitude=latitude, longitude=longitude, pi=pi, sigeo=sigeo)
        self.session.add(loc)
        self.session.commit()
        return loc

    def registrar_apr(self, equipe_id, localizacao_id, riscos, medidas, aprovado_por=None):
        equipe_ok, msg = self.validar_equipe(equipe_id)
        if not equipe_ok:
            raise ValueError(f"Não é possível gerar APR: {msg}")

        apr = APR(
            equipe_id=equipe_id,
            localizacao_id=localizacao_id,
            riscos_identificados=riscos,
            medidas_controle=medidas,
            aprovado=bool(aprovado_por),
            aprovado_por=aprovado_por,
        )
        self.session.add(apr)
        self.session.commit()
        registrar_log(self.session, aprovado_por, "APR_REGISTRADA", f"equipe={equipe_id}")
        return apr


# ------------------------------------------------------------
# Exemplo de uso rápido
# ------------------------------------------------------------

init_db()  # cria as novas tabelas também

seg = SegurancaService(db)  # reaproveita a "db" da Parte 1

equipe = seg.criar_equipe("Equipe Alfa", responsavel_id=novo.id, membros_ids=[novo.id])
seg.emitir_certificacao(novo.id, "NR-10")

ok, msg = seg.validar_equipe(equipe.id)
print("Equipe válida?", ok, "-", msg)

loc = seg.cadastrar_localizacao("Rua das Flores, 123", -23.55, -46.63, pi="PI-0001", sigeo="SIGEO-9988")

apr = seg.registrar_apr(
    equipe.id, loc.id,
    riscos="Trabalho em altura, rede energizada",
    medidas="Uso de EPI, bloqueio e etiquetagem",
    aprovado_por=novo.id
)
print("APR criada:", apr.id, "- aprovada:", apr.aprovado)

Equipe válida? True - Equipe validada
APR criada: f2919b97-065d-41d3-a498-818935eae780 - aprovada: True


/tmp/ipykernel_3895/698150147.py:158: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  equipe = self.session.query(Equipe).get(equipe_id)
/tmp/ipykernel_3895/698150147.py:68: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow() <= self.data_expiracao


programação

In [ ]:
# ============================================================
# ELETROGESTOR 2.0 - PARTE 3/N: PROGRAMAÇÃO
# (rode as Partes 1 e 2 antes, na mesma sessão do Colab)
# ============================================================

# ------------------------------------------------------------
# PROGRAMAÇÃO > STATUS DA OBRA
# ------------------------------------------------------------

class StatusObra(str, Enum):
    BACKLOG = "backlog"           # aguardando programação
    PROGRAMADA = "programada"     # data/equipe definidas
    LIBERADA = "liberada"         # APR + equipe validadas, kit gerado
    EM_EXECUCAO = "em_execucao"
    CONCLUIDA = "concluida"
    CANCELADA = "cancelada"
    MANUTENCAO = "manutencao"     # retrabalho / manutenção


# ------------------------------------------------------------
# PROGRAMAÇÃO > OBRAS
# ------------------------------------------------------------

class Obra(Base):
    __tablename__ = "obras"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    codigo = Column(String(50), unique=True, nullable=False)
    descricao = Column(String(255), nullable=False)
    localizacao_id = Column(String(36), ForeignKey("localizacoes.id"), nullable=True)
    equipe_id = Column(String(36), ForeignKey("equipes.id"), nullable=True)
    apr_id = Column(String(36), ForeignKey("aprs.id"), nullable=True)

    status = Column(SAEnum(StatusObra), default=StatusObra.BACKLOG)
    prioridade = Column(Integer, default=3)  # 1 (urgente) a 5 (baixa)

    data_programada = Column(DateTime, nullable=True)
    data_inicio_execucao = Column(DateTime, nullable=True)
    data_conclusao = Column(DateTime, nullable=True)

    criado_em = Column(DateTime, default=datetime.utcnow)
    atualizado_em = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    localizacao = relationship("Localizacao")
    equipe = relationship("Equipe")
    apr = relationship("APR")
    historico = relationship("HistoricoObra", back_populates="obra")


class HistoricoObra(Base):
    __tablename__ = "historico_obras"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    obra_id = Column(String(36), ForeignKey("obras.id"), nullable=False)
    status_anterior = Column(SAEnum(StatusObra), nullable=True)
    status_novo = Column(SAEnum(StatusObra), nullable=False)
    observacao = Column(Text, nullable=True)
    usuario_id = Column(String(36), ForeignKey("usuarios.id"), nullable=True)
    criado_em = Column(DateTime, default=datetime.utcnow)

    obra = relationship("Obra", back_populates="historico")


# ------------------------------------------------------------
# PROGRAMAÇÃO > SERVIÇO
# ------------------------------------------------------------

class ProgramacaoService:
    def __init__(self, session):
        self.session = session
        self.seg = SegurancaService(session)

    # ---- Backlog / Carteira ----
    def criar_obra(self, codigo, descricao, localizacao_id=None, prioridade=3):
        obra = Obra(
            codigo=codigo, descricao=descricao,
            localizacao_id=localizacao_id, prioridade=prioridade,
            status=StatusObra.BACKLOG,
        )
        self.session.add(obra)
        self.session.commit()
        self._registrar_transicao(obra, None, StatusObra.BACKLOG, "Obra criada")
        return obra

    def carteira(self, status=None):
        """Retorna obras filtradas por status (carteira geral se status=None)."""
        q = self.session.query(Obra)
        if status:
            q = q.filter_by(status=status)
        return q.order_by(Obra.prioridade, Obra.criado_em).all()

    def backlog(self):
        return self.carteira(StatusObra.BACKLOG)

    # ---- Programação (Obras > Programação) ----
    def programar_obra(self, obra_id, equipe_id, data_programada):
        obra = self.session.query(Obra).get(obra_id)
        if not obra:
            raise ValueError("Obra não encontrada")
        if obra.status != StatusObra.BACKLOG:
            raise ValueError(f"Obra não está em backlog (status atual: {obra.status})")

        equipe_ok, msg = self.seg.validar_equipe(equipe_id)
        if not equipe_ok:
            raise ValueError(f"Equipe não pode ser alocada: {msg}")

        status_anterior = obra.status
        obra.equipe_id = equipe_id
        obra.data_programada = data_programada
        obra.status = StatusObra.PROGRAMADA
        self.session.commit()

        self._registrar_transicao(obra, status_anterior, StatusObra.PROGRAMADA, f"Equipe {equipe_id}")
        return obra

    # ---- Localização + Pré-APR -> Liberação ----
    def liberar_obra(self, obra_id, apr_id):
        obra = self.session.query(Obra).get(obra_id)
        if not obra:
            raise ValueError("Obra não encontrada")
        if obra.status != StatusObra.PROGRAMADA:
            raise ValueError("Obra precisa estar programada antes de liberar")

        apr = self.session.query(APR).get(apr_id)
        if not apr or not apr.aprovado:
            raise ValueError("APR não aprovada")

        status_anterior = obra.status
        obra.apr_id = apr_id
        obra.status = StatusObra.LIBERADA
        self.session.commit()

        self._registrar_transicao(obra, status_anterior, StatusObra.LIBERADA, "APR aprovada, kit pode ser gerado")
        return obra

    # ---- Execução (Monitoramento) ----
    def iniciar_execucao(self, obra_id):
        obra = self.session.query(Obra).get(obra_id)
        if obra.status != StatusObra.LIBERADA:
            raise ValueError("Obra precisa estar liberada para iniciar execução")

        status_anterior = obra.status
        obra.status = StatusObra.EM_EXECUCAO
        obra.data_inicio_execucao = datetime.utcnow()
        self.session.commit()

        self._registrar_transicao(obra, status_anterior, StatusObra.EM_EXECUCAO)
        return obra

    def concluir_obra(self, obra_id, observacao=None):
        obra = self.session.query(Obra).get(obra_id)
        if obra.status != StatusObra.EM_EXECUCAO:
            raise ValueError("Obra precisa estar em execução para concluir")

        status_anterior = obra.status
        obra.status = StatusObra.CONCLUIDA
        obra.data_conclusao = datetime.utcnow()
        self.session.commit()

        self._registrar_transicao(obra, status_anterior, StatusObra.CONCLUIDA, observacao)
        return obra

    def enviar_manutencao(self, obra_id, motivo):
        obra = self.session.query(Obra).get(obra_id)
        status_anterior = obra.status
        obra.status = StatusObra.MANUTENCAO
        self.session.commit()
        self._registrar_transicao(obra, status_anterior, StatusObra.MANUTENCAO, motivo)
        return obra

    # ---- Agenda Inteligente (sugestão simples de ordenação por prioridade/data) ----
    def agenda_inteligente(self, data_inicio=None, data_fim=None):
        q = self.session.query(Obra).filter(Obra.status == StatusObra.PROGRAMADA)
        if data_inicio:
            q = q.filter(Obra.data_programada >= data_inicio)
        if data_fim:
            q = q.filter(Obra.data_programada <= data_fim)
        return q.order_by(Obra.prioridade, Obra.data_programada).all()

    # ---- Dash (indicadores simples) ----
    def dash_resumo(self):
        resumo = {}
        for status in StatusObra:
            resumo[status.value] = self.session.query(Obra).filter_by(status=status).count()
        return resumo

    # ---- interno ----
    def _registrar_transicao(self, obra, status_anterior, status_novo, observacao=None, usuario_id=None):
        hist = HistoricoObra(
            obra_id=obra.id, status_anterior=status_anterior,
            status_novo=status_novo, observacao=observacao, usuario_id=usuario_id,
        )
        self.session.add(hist)
        self.session.commit()


# ------------------------------------------------------------
# Exemplo de uso rápido
# ------------------------------------------------------------

init_db()

prog = ProgramacaoService(db)  # reaproveita "db" das partes anteriores

obra = prog.criar_obra("OBRA-0001", "Substituição de poste - Rede BT", localizacao_id=loc.id, prioridade=1)
print("Obra criada:", obra.codigo, "- status:", obra.status)

obra = prog.programar_obra(obra.id, equipe.id, datetime.utcnow() + timedelta(days=1))
print("Obra programada para:", obra.data_programada)

obra = prog.liberar_obra(obra.id, apr.id)
print("Obra liberada:", obra.status)

obra = prog.iniciar_execucao(obra.id)
obra = prog.concluir_obra(obra.id, observacao="Execução sem intercorrências")
print("Obra concluída em:", obra.data_conclusao)

print("Dash resumo:", prog.dash_resumo())

/tmp/ipykernel_3895/882923551.py:208: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  obra = prog.programar_obra(obra.id, equipe.id, datetime.utcnow() + timedelta(days=1))
/tmp/ipykernel_3895/882923551.py:97: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  obra = self.session.query(Obra).get(obra_id)
/tmp/ipykernel_3895/698150147.py:158: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  equipe = self

Obra criada: OBRA-0001 - status: StatusObra.BACKLOG
Obra programada para: 2026-09-09 13:22:15.530280
Obra liberada: StatusObra.LIBERADA
Obra concluída em: 2026-09-08 13:22:15.602378
Dash resumo: {'backlog': 0, 'programada': 0, 'liberada': 0, 'em_execucao': 0, 'concluida': 1, 'cancelada': 0, 'manutencao': 0}


/tmp/ipykernel_3895/882923551.py:151: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  obra = self.session.query(Obra).get(obra_id)
/tmp/ipykernel_3895/882923551.py:157: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  obra.data_conclusao = datetime.utcnow()


area de gerencia geral eletro gestor

In [ ]:
# -*- coding: utf-8 -*-
"""
EletroGestor - Gerência Geral
Automações a implementar agora (lógica em Python, testável)

Este script implementa, em forma de funções puras e testáveis, as automações
descritas no documento "Automações a implementar agora":

  1. Financeiro
  2. Cronograma
  3. Segurança do trabalho (consolidado)
  4. Comercial / Pipeline

Princípio geral respeitado no código: a Gerência Geral é 100% LEITURA.
As funções abaixo apenas CONSOMEM dados que, no sistema real, viriam dos
módulos de Programação, Campo, Segurança e Faturamento (aqui simulados por
listas de dicionários em memória, para permitir teste imediato).

Como testar:
  - Rode o arquivo direto: python gerencia_geral_automacoes.py
  - Ele imprime o dashboard consolidado com os dados de exemplo (MOCK_OBRAS,
    MOCK_MARCOS, MOCK_TRABALHADORES, MOCK_STOP_WORK).
  - No final, pede um input opcional: o código da obra para "furar"
    (drill-down) e ver o detalhe. Enter em branco pula essa etapa.
  - Depois de validar a lógica aqui, ela pode ser copiada de volta para o
    NotebookLM como evidência de implementação/especificação técnica.
"""

from __future__ import annotations
from dataclasses import dataclass, field
from datetime import date, timedelta
from enum import Enum
from typing import Optional


# ---------------------------------------------------------------------------
# 0. MODELOS DE DADOS (viriam do banco / outros módulos no sistema real)
# ---------------------------------------------------------------------------

class StatusObra(Enum):
    PROPOSTA = "proposta"
    CONTRATADA = "contratada"
    EM_EXECUCAO = "em_execucao"
    CONCLUIDA = "concluida"


class StatusSemaforo(Enum):
    OK = "ok"
    ATENCAO = "atenção"
    CRITICO = "crítico"


@dataclass
class Obra:
    codigo: str
    nome: str
    equipe: str
    status: StatusObra
    valor_contrato: float
    custo_previsto: float
    custo_realizado: float
    avanco_previsto_pct: float   # 0-100
    avanco_realizado_pct: float  # 0-100
    data_inicio_execucao: Optional[date] = None
    data_contratacao: Optional[date] = None


@dataclass
class Marco:
    obra_codigo: str
    nome: str
    data_prevista: date
    data_conclusao: Optional[date]  # None = ainda não concluído
    faturado: bool = False


@dataclass
class Trabalhador:
    nome: str
    equipe: str
    aso_validade: date
    nr10_validade: Optional[date]
    nr35_validade: Optional[date]


@dataclass
class StopWork:
    obra_codigo: str
    motivo: str
    data_parada: date
    data_retomada: Optional[date]  # None = ainda parado


@dataclass
class IncidenteRegistravel:
    obra_codigo: str
    data: date
    descricao: str


@dataclass
class ScoreEquipe:
    equipe: str
    nota_prazo: float      # 1 a 5
    nota_qualidade: float  # 1 a 5
    nota_seguranca: float  # 1 a 5


HOJE = date.today()


# ---------------------------------------------------------------------------
# 1. FINANCEIRO
# ---------------------------------------------------------------------------

def desvio_orcamentario(obra: Obra) -> dict:
    """Previsto x realizado, com status ok/atenção/crítico."""
    if obra.custo_previsto == 0:
        desvio_pct = 0.0
    else:
        desvio_pct = ((obra.custo_realizado - obra.custo_previsto) / obra.custo_previsto) * 100

    if desvio_pct <= 5:
        status = StatusSemaforo.OK
    elif desvio_pct <= 15:
        status = StatusSemaforo.ATENCAO
    else:
        status = StatusSemaforo.CRITICO

    return {
        "obra": obra.codigo,
        "previsto": obra.custo_previsto,
        "realizado": obra.custo_realizado,
        "desvio_pct": round(desvio_pct, 1),
        "status": status.value,
    }


def projecao_custo_final(obra: Obra) -> dict:
    """
    Projeta o custo final com base no ritmo de gasto atual x avanço físico.
    Regra: custo_projetado = custo_realizado / (avanco_realizado_pct / 100)
    (assume que o ritmo de gasto por % de avanço se mantém constante).
    """
    if obra.avanco_realizado_pct <= 0:
        custo_projetado = obra.custo_previsto  # sem avanço ainda, usa o previsto
    else:
        custo_projetado = obra.custo_realizado / (obra.avanco_realizado_pct / 100)

    estouro = custo_projetado - obra.custo_previsto
    return {
        "obra": obra.codigo,
        "custo_projetado": round(custo_projetado, 2),
        "estouro_previsto": round(estouro, 2),
        "status": (
            StatusSemaforo.CRITICO.value if estouro > obra.custo_previsto * 0.15
            else StatusSemaforo.ATENCAO.value if estouro > 0
            else StatusSemaforo.OK.value
        ),
    }


def margem_obra(obra: Obra) -> dict:
    """Margem = (valor do contrato - custo realizado) / valor do contrato."""
    if obra.valor_contrato == 0:
        margem_pct = 0.0
    else:
        margem_pct = ((obra.valor_contrato - obra.custo_realizado) / obra.valor_contrato) * 100
    return {"obra": obra.codigo, "margem_pct": round(margem_pct, 1)}


def pendencias_faturamento(marcos: list[Marco]) -> list[dict]:
    """Todo marco concluído e não faturado -> sugestão automática de medição."""
    pendentes = [m for m in marcos if m.data_conclusao is not None and not m.faturado]
    return [
        {
            "obra": m.obra_codigo,
            "marco": m.nome,
            "concluido_em": m.data_conclusao.isoformat(),
            "sugestao": "gerar medição de faturamento",
        }
        for m in pendentes
    ]


# ---------------------------------------------------------------------------
# 2. CRONOGRAMA
# ---------------------------------------------------------------------------

def avanco_obra(obra: Obra) -> dict:
    return {
        "obra": obra.codigo,
        "previsto_pct": obra.avanco_previsto_pct,
        "realizado_pct": obra.avanco_realizado_pct,
        "diferenca_pct": round(obra.avanco_realizado_pct - obra.avanco_previsto_pct, 1),
    }


def dias_atraso_por_marco(marcos: list[Marco]) -> list[dict]:
    """Dias de atraso = hoje - data_prevista, apenas para marcos ainda não concluídos e já vencidos."""
    resultado = []
    for m in marcos:
        if m.data_conclusao is None and m.data_prevista < HOJE:
            atraso = (HOJE - m.data_prevista).days
            resultado.append({"obra": m.obra_codigo, "marco": m.nome, "dias_atraso": atraso})
    return resultado


def marcos_concluidos_recentemente(marcos: list[Marco], janela_dias: int = 7) -> list[dict]:
    limite = HOJE - timedelta(days=janela_dias)
    recentes = [
        m for m in marcos
        if m.data_conclusao is not None and m.data_conclusao >= limite
    ]
    return [
        {"obra": m.obra_codigo, "marco": m.nome, "concluido_em": m.data_conclusao.isoformat()}
        for m in recentes
    ]


def alerta_preditivo_atraso(obra: Obra, ritmo_medio_diario_pct: float = 0.5) -> dict:
    """
    Estima quantos dias faltam, no ritmo atual, para concluir os 100% e
    compara com o quanto falta segundo o cronograma previsto.
    ritmo_medio_diario_pct: configurável, % de avanço que a equipe entrega por dia (mock).
    """
    falta_pct = max(0.0, 100 - obra.avanco_realizado_pct)
    if ritmo_medio_diario_pct <= 0:
        dias_estimados = None
    else:
        dias_estimados = falta_pct / ritmo_medio_diario_pct

    risco = dias_estimados is not None and dias_estimados > 30  # limiar configurável
    return {
        "obra": obra.codigo,
        "falta_pct": round(falta_pct, 1),
        "dias_estimados_para_100pct": round(dias_estimados, 1) if dias_estimados is not None else None,
        "risco_atraso": risco,
    }


# ---------------------------------------------------------------------------
# 3. SEGURANÇA DO TRABALHO (CONSOLIDADO)
# ---------------------------------------------------------------------------

def incidentes_e_dias_sem_incidente(obra_codigo: str, incidentes: list[IncidenteRegistravel]) -> dict:
    do_obra = sorted((i for i in incidentes if i.obra_codigo == obra_codigo), key=lambda i: i.data)
    if not do_obra:
        return {"obra": obra_codigo, "total_incidentes": 0, "dias_sem_incidente": None}
    ultimo = do_obra[-1].data
    return {
        "obra": obra_codigo,
        "total_incidentes": len(do_obra),
        "dias_sem_incidente": (HOJE - ultimo).days,
    }


def historico_stop_work(stop_work: list[StopWork]) -> list[dict]:
    resultado = []
    for sw in stop_work:
        status = "retomado" if sw.data_retomada else "parado"
        resultado.append({
            "obra": sw.obra_codigo,
            "motivo": sw.motivo,
            "data_parada": sw.data_parada.isoformat(),
            "data_retomada": sw.data_retomada.isoformat() if sw.data_retomada else None,
            "status": status,
        })
    return resultado


def bloqueio_alocacao_documento_vencido(trabalhadores: list[Trabalhador]) -> list[dict]:
    """
    Bloqueia (sinaliza) trabalhador com ASO, NR-10 ou NR-35 vencidos.
    NR-10/NR-35 são opcionais por trabalhador (nem todo cargo exige as duas).
    """
    bloqueados = []
    for t in trabalhadores:
        motivos = []
        if t.aso_validade < HOJE:
            motivos.append("ASO vencido")
        if t.nr10_validade is not None and t.nr10_validade < HOJE:
            motivos.append("NR-10 vencida")
        if t.nr35_validade is not None and t.nr35_validade < HOJE:
            motivos.append("NR-35 vencida")
        if motivos:
            bloqueados.append({
                "trabalhador": t.nome,
                "equipe": t.equipe,
                "bloqueado": True,
                "motivos": motivos,
            })
    return bloqueados


def caidi_proxy(tempos_restabelecimento_horas: list[float]) -> dict:
    """
    Indicador já existente: tempo médio de restabelecimento (proxy de CAIDI).
    Recebe uma lista de tempos (em horas) de restabelecimento e retorna a média.
    """
    if not tempos_restabelecimento_horas:
        return {"caidi_proxy_horas": None, "amostras": 0}
    media = sum(tempos_restabelecimento_horas) / len(tempos_restabelecimento_horas)
    return {"caidi_proxy_horas": round(media, 2), "amostras": len(tempos_restabelecimento_horas)}


# ---------------------------------------------------------------------------
# 4. COMERCIAL / PIPELINE
# ---------------------------------------------------------------------------

def valor_em_carteira_por_status(obras: list[Obra]) -> dict:
    breakdown: dict[str, float] = {}
    for status in StatusObra:
        breakdown[status.value] = sum(o.valor_contrato for o in obras if o.status == status)
    breakdown["total"] = sum(breakdown[s.value] for s in StatusObra)
    return breakdown


def gargalo_contratadas_aguardando_inicio(obras: list[Obra], limite_dias: int = 15) -> list[dict]:
    alertas = []
    for o in obras:
        if o.status == StatusObra.CONTRATADA and o.data_contratacao:
            dias_esperando = (HOJE - o.data_contratacao).days
            if dias_esperando > limite_dias:
                alertas.append({
                    "obra": o.codigo,
                    "dias_aguardando_inicio": dias_esperando,
                    "limite_configurado": limite_dias,
                    "alerta": "gargalo",
                })
    return alertas


def scorecard_mensal_equipes(scores: list[ScoreEquipe]) -> list[dict]:
    resultado = []
    for s in scores:
        nota_consolidada = round((s.nota_prazo + s.nota_qualidade + s.nota_seguranca) / 3, 2)
        resultado.append({
            "equipe": s.equipe,
            "nota_prazo": s.nota_prazo,
            "nota_qualidade": s.nota_qualidade,
            "nota_seguranca": s.nota_seguranca,
            "nota_consolidada": nota_consolidada,
        })
    return sorted(resultado, key=lambda r: r["nota_consolidada"], reverse=True)


# ---------------------------------------------------------------------------
# 5. DASHBOARD CONSOLIDADO (o que a Gerência Geral efetivamente vê)
# ---------------------------------------------------------------------------

def montar_dashboard(
    obras: list[Obra],
    marcos: list[Marco],
    trabalhadores: list[Trabalhador],
    stop_work: list[StopWork],
    incidentes: list[IncidenteRegistravel],
    scores: list[ScoreEquipe],
    tempos_restabelecimento: list[float],
) -> dict:
    return {
        "financeiro": {
            "desvio_orcamentario": [desvio_orcamentario(o) for o in obras],
            "projecao_custo_final": [projecao_custo_final(o) for o in obras],
            "margem_por_obra": [margem_obra(o) for o in obras],
            "pendencias_faturamento": pendencias_faturamento(marcos),
        },
        "cronograma": {
            "avanco_por_obra": [avanco_obra(o) for o in obras],
            "dias_atraso_por_marco": dias_atraso_por_marco(marcos),
            "marcos_concluidos_recentemente": marcos_concluidos_recentemente(marcos),
            "alerta_preditivo": [alerta_preditivo_atraso(o) for o in obras],
        },
        "seguranca": {
            "por_obra": [
                incidentes_e_dias_sem_incidente(o.codigo, incidentes) for o in obras
            ],
            "stop_work": historico_stop_work(stop_work),
            "trabalhadores_bloqueados": bloqueio_alocacao_documento_vencido(trabalhadores),
            "caidi_proxy": caidi_proxy(tempos_restabelecimento),
        },
        "comercial": {
            "valor_em_carteira": valor_em_carteira_por_status(obras),
            "gargalos_inicio_execucao": gargalo_contratadas_aguardando_inicio(obras),
            "scorecard_equipes": scorecard_mensal_equipes(scores),
        },
    }


def imprimir_dashboard(dash: dict) -> None:
    print("\n=================== GERÊNCIA GERAL — DASHBOARD ===================\n")

    print("--- 1. FINANCEIRO ---")
    for d in dash["financeiro"]["desvio_orcamentario"]:
        print(f"  Obra {d['obra']}: previsto R${d['previsto']:.2f} | realizado R${d['realizado']:.2f} "
              f"| desvio {d['desvio_pct']}% | status: {d['status'].upper()}")
    for p in dash["financeiro"]["projecao_custo_final"]:
        print(f"  Obra {p['obra']}: custo projetado R${p['custo_projetado']:.2f} "
              f"(estouro R${p['estouro_previsto']:.2f}) | status: {p['status'].upper()}")
    for m in dash["financeiro"]["margem_por_obra"]:
        print(f"  Obra {m['obra']}: margem {m['margem_pct']}%")
    if dash["financeiro"]["pendencias_faturamento"]:
        print("  Pendências de faturamento:")
        for pf in dash["financeiro"]["pendencias_faturamento"]:
            print(f"    - Obra {pf['obra']} | marco '{pf['marco']}' concluído em {pf['concluido_em']} "
                  f"-> {pf['sugestao']}")
    else:
        print("  Sem pendências de faturamento.")

    print("\n--- 2. CRONOGRAMA ---")
    for a in dash["cronograma"]["avanco_por_obra"]:
        print(f"  Obra {a['obra']}: previsto {a['previsto_pct']}% | realizado {a['realizado_pct']}% "
              f"| diferença {a['diferenca_pct']} p.p.")
    if dash["cronograma"]["dias_atraso_por_marco"]:
        print("  Marcos atrasados:")
        for da in dash["cronograma"]["dias_atraso_por_marco"]:
            print(f"    - Obra {da['obra']} | marco '{da['marco']}' | {da['dias_atraso']} dias de atraso")
    else:
        print("  Nenhum marco atrasado.")
    if dash["cronograma"]["marcos_concluidos_recentemente"]:
        print("  Marcos concluídos recentemente:")
        for mc in dash["cronograma"]["marcos_concluidos_recentemente"]:
            print(f"    - Obra {mc['obra']} | '{mc['marco']}' em {mc['concluido_em']}")
    for al in dash["cronograma"]["alerta_preditivo"]:
        risco_txt = "RISCO DE ATRASO" if al["risco_atraso"] else "dentro do esperado"
        print(f"  Obra {al['obra']}: falta {al['falta_pct']}% | ~{al['dias_estimados_para_100pct']} dias "
              f"para 100% | {risco_txt}")

    print("\n--- 3. SEGURANÇA DO TRABALHO ---")
    for s in dash["seguranca"]["por_obra"]:
        print(f"  Obra {s['obra']}: {s['total_incidentes']} incidente(s) | "
              f"dias sem incidente: {s['dias_sem_incidente']}")
    if dash["seguranca"]["stop_work"]:
        print("  Histórico de Stop Work:")
        for sw in dash["seguranca"]["stop_work"]:
            print(f"    - Obra {sw['obra']} | motivo: {sw['motivo']} | status: {sw['status']}")
    if dash["seguranca"]["trabalhadores_bloqueados"]:
        print("  Trabalhadores bloqueados por documento vencido:")
        for b in dash["seguranca"]["trabalhadores_bloqueados"]:
            print(f"    - {b['trabalhador']} ({b['equipe']}): {', '.join(b['motivos'])}")
    else:
        print("  Nenhum trabalhador bloqueado.")
    caidi = dash["seguranca"]["caidi_proxy"]
    print(f"  CAIDI (proxy) tempo médio de restabelecimento: {caidi['caidi_proxy_horas']} h "
          f"({caidi['amostras']} amostra(s))")

    print("\n--- 4. COMERCIAL / PIPELINE ---")
    carteira = dash["comercial"]["valor_em_carteira"]
    for k, v in carteira.items():
        print(f"  {k}: R${v:,.2f}")
    if dash["comercial"]["gargalos_inicio_execucao"]:
        print("  Gargalos (contratadas aguardando início):")
        for g in dash["comercial"]["gargalos_inicio_execucao"]:
            print(f"    - Obra {g['obra']} | aguardando há {g['dias_aguardando_inicio']} dias "
                  f"(limite: {g['limite_configurado']})")
    else:
        print("  Nenhum gargalo de início de execução.")
    print("  Scorecard mensal por equipe:")
    for sc in dash["comercial"]["scorecard_equipes"]:
        print(f"    - {sc['equipe']}: prazo {sc['nota_prazo']} | qualidade {sc['nota_qualidade']} "
              f"| segurança {sc['nota_seguranca']} | CONSOLIDADA {sc['nota_consolidada']}")

    print("\n====================================================================\n")


# ---------------------------------------------------------------------------
# 6. DADOS DE TESTE (MOCK) — no sistema real vêm de Programação/Campo/Faturamento
# ---------------------------------------------------------------------------

MOCK_OBRAS = [
    Obra(
        codigo="OBR-001", nome="Rede MT - Trecho A", equipe="Equipe Alfa",
        status=StatusObra.EM_EXECUCAO, valor_contrato=500_000.0,
        custo_previsto=350_000.0, custo_realizado=390_000.0,
        avanco_previsto_pct=70.0, avanco_realizado_pct=60.0,
    ),
    Obra(
        codigo="OBR-002", nome="Poste e Transformador - Zona Sul", equipe="Equipe Beta",
        status=StatusObra.EM_EXECUCAO, valor_contrato=120_000.0,
        custo_previsto=90_000.0, custo_realizado=88_000.0,
        avanco_previsto_pct=50.0, avanco_realizado_pct=55.0,
    ),
    Obra(
        codigo="OBR-003", nome="Extensão de rede rural", equipe="Equipe Gama",
        status=StatusObra.CONTRATADA, valor_contrato=200_000.0,
        custo_previsto=150_000.0, custo_realizado=0.0,
        avanco_previsto_pct=0.0, avanco_realizado_pct=0.0,
        data_contratacao=HOJE - timedelta(days=25),
    ),
    Obra(
        codigo="OBR-004", nome="Manutenção preventiva - BT", equipe="Equipe Alfa",
        status=StatusObra.CONCLUIDA, valor_contrato=80_000.0,
        custo_previsto=60_000.0, custo_realizado=58_000.0,
        avanco_previsto_pct=100.0, avanco_realizado_pct=100.0,
    ),
]

MOCK_MARCOS = [
    Marco("OBR-001", "Escavação concluída", HOJE - timedelta(days=10),
          data_conclusao=HOJE - timedelta(days=8), faturado=False),
    Marco("OBR-001", "Lançamento de cabos", HOJE - timedelta(days=2),
          data_conclusao=None, faturado=False),  # vencido, não concluído -> atraso
    Marco("OBR-002", "Instalação do transformador", HOJE - timedelta(days=1),
          data_conclusao=HOJE - timedelta(days=1), faturado=True),
    Marco("OBR-004", "Entrega final", HOJE - timedelta(days=20),
          data_conclusao=HOJE - timedelta(days=20), faturado=False),  # pendente de faturamento
]

MOCK_TRABALHADORES = [
    Trabalhador("João Silva", "Equipe Alfa", aso_validade=HOJE + timedelta(days=30),
                nr10_validade=HOJE - timedelta(days=5), nr35_validade=HOJE + timedelta(days=90)),
    Trabalhador("Maria Souza", "Equipe Beta", aso_validade=HOJE - timedelta(days=3),
                nr10_validade=None, nr35_validade=None),
    Trabalhador("Carlos Lima", "Equipe Gama", aso_validade=HOJE + timedelta(days=60),
                nr10_validade=HOJE + timedelta(days=60), nr35_validade=HOJE + timedelta(days=60)),
]

MOCK_STOP_WORK = [
    StopWork("OBR-001", "Falta de EPI adequado", HOJE - timedelta(days=6),
             data_retomada=HOJE - timedelta(days=5)),
    StopWork("OBR-002", "Condição climática de risco", HOJE - timedelta(days=1),
             data_retomada=None),
]

MOCK_INCIDENTES = [
    IncidenteRegistravel("OBR-001", HOJE - timedelta(days=40), "Quase-acidente com ferramenta"),
]

MOCK_SCORES = [
    ScoreEquipe("Equipe Alfa", nota_prazo=3.5, nota_qualidade=4.5, nota_seguranca=4.0),
    ScoreEquipe("Equipe Beta", nota_prazo=4.5, nota_qualidade=4.0, nota_seguranca=5.0),
    ScoreEquipe("Equipe Gama", nota_prazo=5.0, nota_qualidade=5.0, nota_seguranca=5.0),
]

MOCK_TEMPOS_RESTABELECIMENTO = [2.5, 3.0, 1.8, 4.2]  # horas


# ---------------------------------------------------------------------------
# 7. DRILL-DOWN (furar até a obra específica)
# ---------------------------------------------------------------------------

def drill_down_obra(codigo_obra: str) -> None:
    obra = next((o for o in MOCK_OBRAS if o.codigo == codigo_obra), None)
    if obra is None:
        print(f"\n[Drill-down] Obra '{codigo_obra}' não encontrada.\n")
        return

    print(f"\n--------- Drill-down: {obra.codigo} - {obra.nome} ---------")
    print(f"  Equipe: {obra.equipe} | Status: {obra.status.value}")
    print(f"  Financeiro: {desvio_orcamentario(obra)}")
    print(f"  Projeção: {projecao_custo_final(obra)}")
    print(f"  Margem: {margem_obra(obra)}")
    print(f"  Cronograma: {avanco_obra(obra)}")
    print(f"  Alerta preditivo: {alerta_preditivo_atraso(obra)}")
    marcos_da_obra = [m for m in MOCK_MARCOS if m.obra_codigo == obra.codigo]
    print(f"  Marcos da obra: {[m.nome for m in marcos_da_obra]}")
    print("----------------------------------------------------------\n")


# ---------------------------------------------------------------------------
# 8. EXECUÇÃO / TESTE MANUAL
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    dashboard = montar_dashboard(
        obras=MOCK_OBRAS,
        marcos=MOCK_MARCOS,
        trabalhadores=MOCK_TRABALHADORES,
        stop_work=MOCK_STOP_WORK,
        incidentes=MOCK_INCIDENTES,
        scores=MOCK_SCORES,
        tempos_restabelecimento=MOCK_TEMPOS_RESTABELECIMENTO,
    )
    imprimir_dashboard(dashboard)

    try:
        codigo = input("Digite o código de uma obra para 'furar' (drill-down) ou Enter para pular: ").strip()
    except EOFError:
        codigo = ""

    if codigo:
        drill_down_obra(codigo)
    else:
        print("Nenhum drill-down solicitado. Fim da execução de teste.\n")


=== BADGES DAS MINI ABAS ===
{
  "financeiro": 0,
  "cronograma": 3,
  "seguranca": 2,
  "comercial": 1
}

=== MINI ABA: FINANCEIRO ===
{
  "titulo": "Financeiro",
  "badge_alertas": 0,
  "desvio_orcamentario": [
    {
      "obra_id": "OBRA-01",
      "obra_nome": "Rede Bairro Norte",
      "valor_contrato": 500000,
      "custo_atual": 430000,
      "desvio_percentual": -14.0,
      "status": "ok"
    },
    {
      "obra_id": "OBRA-02",
      "obra_nome": "Subestação Sul",
      "valor_contrato": 1200000,
      "custo_atual": 1150000,
      "desvio_percentual": -4.17,
      "status": "ok"
    },
    {
      "obra_id": "OBRA-03",
      "obra_nome": "Ramal Zona Leste",
      "valor_contrato": 300000,
      "custo_atual": 0,
      "desvio_percentual": -100.0,
      "status": "ok"
    },
    {
      "obra_id": "OBRA-04",
      "obra_nome": "Extensão Distrito Industrial",
      "valor_contrato": 800000,
      "custo_atual": 0,
      "desvio_percentual": -100.0,
      "status": "ok"
    